### 1. Installations:

In [ ]:
!pip install openai

In [ ]:
!pip install ipython-autotime
%load_ext autotime

In [ ]:
!pip install datasets

In [ ]:
from google.colab import userdata
hf= userdata.get('hf_new')
myGPT = userdata.get('MyGPT')

### Paraphrase Module:

In [ ]:
import pandas as pd
test= pd.read_csv("/content/test2.csv")
#add index column:
test.insert(0, 'index', range(1, len(test) + 1))
test['dataset'] = 'm-absa'

test

In [ ]:
import pandas as pd
import os
from openai import OpenAI
from time import sleep
from google.colab import drive

# # 1. Mount Google Drive
drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/colab_files/m-absa-English-test.csv'

# 2. Load from saved file if available (to resume)
if os.path.exists(save_path):
    paraphrased_df = pd.read_csv(save_path)
    done_indices = set(paraphrased_df['index'])
else:
    paraphrased_df = pd.DataFrame(columns=['index', 'dataset', 'domain', 'sentence', 'aspect', 'category',  'type', 'source_dataset','language', 'paraphrases'])
    done_indices = set()

client = OpenAI(api_key=myGPT)

def paraphrase_text(text):
    for _ in range(3):  # Retry in case of temporary API errors
        try:
            response = client.chat.completions.create(
                messages=[{
                    "role": "user",
                    "content": f"Act as an expert linguistic in English Language, Give 10 different paraphrases to the following text in English. trying to explicitly state every noun described in the sentence. Only include the paraphrased sentence with no additional text before or after: {text}"
                }],
                model= "gpt-4.1-mini" , #gpt-4.1-mini",
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Error: {e}. Retrying...")
            sleep(5)
    return ""  # Return empty string on failure

# 4. Process and save results incrementally
for idx, row in test.iterrows():
    print("Row", idx)
    print('Paraphrasing:' ,row['text'])
    if idx in done_indices:
        continue  # Skip already done rows
    text = row['text']
    paraphrased = paraphrase_text(text)

    new_row = pd.DataFrame([{
        'index': idx,
        'dataset': row['dataset'],
        'domain': row['domain'],
        'sentence': row['text'],
        'aspect': row['aspect'],
        'category': row['category'],
        'type': row['type'],
        'source_dataset': row['dataset'],
        'language': 'English',
        'paraphrases': paraphrased
    }])
    paraphrased_df = pd.concat([paraphrased_df, new_row], ignore_index=True)

    # Save after every row (you can change this to every 5 or 10 rows)
    paraphrased_df.to_csv(save_path, index=False, encoding='utf-8-sig')


### Post-process paraphrases:

In [ ]:
paraphrased=paraphrased_df.copy()
paraphrased

In [ ]:
''' split the paraphrased text according to newlines, remove any special characters and leading numbers,
Then store every part in a separate column. Make sure to remove any empty lines.
'''

import pandas as pd
import re



def process_text(text):

  # Split the text by newlines
  parts = text.split('\n')

  # Remove special characters from each part and store in a list
  cleaned_parts = []
  for part in parts:
    cleaned_part = re.sub(r'[^\w\s]', '', part).strip()
    cleaned_part = re.sub(r'^\d+', '', cleaned_part).strip() # Remove leading numbers
    #remove â€™ special characters:
    cleaned_part = re.sub(r'â€™', '', cleaned_part).strip()
    if len(cleaned_part)>3:#if the line is non-empty:
      if  'paraphrase' not in cleaned_part and 'sorry' not in cleaned_part:
        cleaned_parts.append(cleaned_part)


  return cleaned_parts


# Apply the function to the 'text' column
paraphrased.loc[:, 'paraphrases']  = paraphrased['paraphrases'].apply(process_text)


In [ ]:
paraphrased.to_csv('m-absa_English_test_10Paraphrases.csv', index=False, encoding='utf-8-sig')
from google.colab import files
files.download('m-absa_English_test_10Paraphrases.csv')